In [1]:
#!pip install gigachat
import json
from gigachat import GigaChat
import pandas as pd

#!pip install razdel
import warnings
warnings.filterwarnings('ignore')


In [ ]:
AUTH_KEY = "=="

In [25]:
data = pd.read_csv('df_for_llm.csv')
data

,doc_id,chunk_id,is_macro,chunk_text
0,0,0,True,"""Индекс Мосбиржи -3%. Это самое большое дневно..."
1,1,0,True,"""​​Циан взлетел на апелляции Крупнейший в Росс..."
2,2,0,True,"""С 5 февраля ЕС вводит эмбарго на российские п..."
3,2,1,True,Госбюджет РФ рискует недополучить еще больше н...
4,3,0,True,"""Новый отчёт Сбера (SBER): чистая прибыль за 1..."
...,...,...,...,...
1949,1491,0,True,"""Финансовый конгресс Банка России: что важного..."
1950,1491,4,True,Этот хаб можно расположить в одной из стран Пе...
1951,1492,0,True,"""За последние 2 недели рубль ослаб почти на 8%..."
1952,1495,2,True,• Нефть дорожает По итогам вчерашних торгов ф...


In [40]:
def label_data(text: str, giga: GigaChat):
    """Классификация текста через GigaChat."""

    # Формируем четкую инструкцию для модели
    prompt2 = ( "Ты — экспертный макроэкономический аналитик. Твоя задача — анализировать новости "
    "и определять динамику 18 показателей. Оценивай прямые упоминания и логические следствия. "
    "Допустимые значения изменений ТОЛЬКО ТАКИЕ: 'up', 'down', 'not stated'. "
    "Ответь строго в формате JSON без вступлений и пояснений: "
    "{"
    "\"процентная ставка\": \"тип изменения\","
    "\"ВВП\": \"тип изменения\","
    "\"инфляция\": \"тип изменения\","
    "\"безработица\": \"тип изменения\","
    "\"капитал\": \"тип изменения\","
    "\"инвестиции\": \"тип изменения\","
    "\"производство\": \"тип изменения\","
    "\"потребление\": \"тип изменения\","
    "\"численность рабочей силы\": \"тип изменения\","
    "\"сбережения\": \"тип изменения\","
    "\"заработные платы\": \"тип изменения\","
    "\"доходы населения\": \"тип изменения\","
    "\"валютный курс\": \"тип изменения\","
    "\"импорт\": \"тип изменения\","
    "\"экспорт\": \"тип изменения\","
    "\"государственные расходы\": \"тип изменения\","
    "\"государственный долг\": \"тип изменения\","
    "\"дефицит бюджета\": \"тип изменения\""
    "} "
    f"Текст новости: {text}"
)
    try:
        response = giga.chat(prompt2)
        content = response.choices[0].message.content

        # Находим JSON в ответе (на случай, если модель добавит лишний текст)
        start = content.find('{')
        end = content.rfind('}') + 1
        return json.loads(content[start:end])
    except Exception as e:
        return {"category": "Ошибка", "details": str(e)}

In [ ]:
## ТЕСТ РАЗМЕТКИ
dataset = [
    """Инфляционные процессы в российской экономике стабилизируются, полагает министр финансов Антон Силаунов. Такое мнение он высказал в письменном заявлении в преддверии заседания международного валютно-финансового комитета. Слова Силуанова приводит пресс-служба Минфина.
    «В российской экономике стабилизируются инфляционные процессы, что является залогом устойчивого и сбалансированного развития в будущем: меры экономической политики, направленные на охлаждение спроса, позволили замедлить инфляцию до минимальных с 2020 года значений – 5,59% по итогам 2025 года. При этом уровень безработицы остается на минимальных значениях (2,1% в феврале 2026 года), создаются рабочие места, растут располагаемые доходы граждан (+7,4% в 2025 году)», - отметил министр.
    По его словам, проводимая бюджетная политика поддерживает устойчивость государственных финансов. Среди стран G20 Россия занимает лидирующее место по уровню государственного долга (16,5% ВВП), а дефицит в размере 2,6% ВВП в 2025 году остается на безопасном уровне.
    «В дальнейшем фокус на укреплении устойчивости сохранится: в условиях нестабильности на сырьевых рынках ключевое внимание будет уделяться сокращению уязвимости бюджета. В этой связи планируется снижение базовой цены на нефть в бюджетном правиле с сопутствующими мероприятиями по консолидации бюджета. Такой подход поможет защитить экономику и бюджет от внешних шоков, укрепит финансовый суверенитет страны и доверие к экономической политике, а также поддержит устойчивое и сбалансированное экономическое развитие», - написал Силуанов""",
    """Банк России находится на финишной прямой перед очередным заседанием по ставке – оно состоится 24 апреля
Аналитики, опрошенные Finam.ru, в основном полагают, что Центробанк снизит ставку на очередные 0,5 п.п., до 14,5%
Влияние на курс рубля будет зависеть от того, насколько сильно решение по ставке и сигнал отклонятся от рыночных ожиданий
На рынке акций на снижение ставки в моменте обычно больше рынка реагируют компании с повышенной долговой нагрузкой
Банк России находится на финишной прямой перед очередным заседанием по ставке – оно состоится 24 апреля. На предыдущем заседании в марте регулятор снизил ставку на 0,5 п.п., до 15%. Накануне предстоящего решения инфляция преподнесла приятный сюрприз: за неделю по 13 апреля она оказалась нулевой после 0,19% на предыдущей неделе. Кроме того, согласно данным опроса «инФОМ», инфляционные ожидания россиян на год вперед в апреле снизились до 12,9% с мартовских 13,4%. Между тем, ВВП страны за два месяца показал падение на 1,8%, на что уже отдельно обращал внимание президент РФ, потребовав предложений для возобновления экономического роста.
Аналитики, опрошенные Finam.ru, в основном полагают, что Центробанк снизит ставку на очередные 0,5 п.п., до 14,5%. Что мешает смягчить политику еще больше, какими будут последствия решения регулятора для рубля и различных классов активов, удастся ли властям не допустить переохлаждения экономики и какую роль в этом могут сыграть действия ЦБ? Finam.ru подготовил обзор.
Смягчение политики ЦБ влечет за собой дальнейшее снижение доходности по депозитам. Однако на рынке облигаций еще остаются возможности зафиксировать высокую доходность на длительный срок.""",
"""Инфляционные ожидания россиян на год вперед в апреле 2026 года снизились до 12,9% с 13,4% в предыдущем месяце, свидетельствуют итоги исследования ООО «инФОМ», опубликованные Банком России.
Показатель опустился ниже 13% впервые с октября 2025 года.
При этом наблюдаемая инфляция в апреле снизилась до 14,6% с 15,6% в марте."""]

# Запуск основной логики
if __name__ == "__main__":
    print("🚀 Начинаю разметку через GigaChat...")

    results = []

    # verify_ssl_certs=False критичен, если у вас нет сертификатов Минцифры
    with GigaChat(credentials=AUTH_KEY, verify_ssl_certs=False) as giga:
        for i, text in enumerate(dataset, 1):
            label = label_data(text, giga)
            label['original_text'] = text
            results.append(label)
    df = pd.DataFrame(results)


In [7]:
df

,category,details,original_text,процентная ставка,ВВП,инфляция,безработица,капитал,инвестиции,производство,...,численность рабочей силы,сбережения,заработные платы,доходы населения,валютный курс,импорт,экспорт,государственные расходы,государственный долг,дефицит бюджета
0,Ошибка,[WinError 10054] Удаленный хост принудительно ...,Инфляционные процессы в российской экономике с...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,Банк России находится на финишной прямой перед...,down,down,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated
2,NaN,NaN,Инфляционные ожидания россиян на год вперед в ...,not stated,not stated,down,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated


## РАЗМЕТИМ ПЕРВЫЕ 100 ПРИМЕРОВ

In [ ]:
print("🚀 Начинаю разметку через GigaChat...")
dataset = list(data['chunk_text'])[:100]
results = []

# verify_ssl_certs=False критичен, если у вас нет сертификатов Минцифры
with GigaChat(credentials=AUTH_KEY, verify_ssl_certs=False) as giga:
    for i, text in enumerate(dataset, 1):
        print(f'Обработка документа {i}...')
        label = label_data(text, giga)
        label['doc_id'] = data.iloc[i, 0]
        label['chunk_id'] = data.iloc[i, 1]
        label['original_text'] = text

        results.append(label)

🚀 Начинаю разметку через GigaChat...
Обработка документа 1...
Обработка документа 2...
Обработка документа 3...
Обработка документа 4...
Обработка документа 5...
Обработка документа 6...
Обработка документа 7...
Обработка документа 8...
Обработка документа 9...
Обработка документа 10...
Обработка документа 11...
Обработка документа 12...
Обработка документа 13...
Обработка документа 14...
Обработка документа 15...
Обработка документа 16...
Обработка документа 17...
Обработка документа 18...
Обработка документа 19...
Обработка документа 20...
Обработка документа 21...
Обработка документа 22...
Обработка документа 23...
Обработка документа 24...
Обработка документа 25...
Обработка документа 26...
Обработка документа 27...
Обработка документа 28...
Обработка документа 29...
Обработка документа 30...
Обработка документа 31...
Обработка документа 32...
Обработка документа 33...
Обработка документа 34...
Обработка документа 35...
Обработка документа 36...
Обработка документа 37...
Обработка 

In [58]:
results

[{'процентная ставка': 'not stated',
  'ВВП': 'not stated',
  'инфляция': 'not stated',
  'безработица': 'not stated',
  'капитал': 'down',
  'инвестиции': 'not stated',
  'производство': 'not stated',
  'потребление': 'not stated',
  'численность рабочей силы': 'not stated',
  'сбережения': 'not stated',
  'заработные платы': 'not stated',
  'доходы населения': 'not stated',
  'валютный курс': 'not stated',
  'импорт': 'not stated',
  'экспорт': 'not stated',
  'государственные расходы': 'not stated',
  'государственный долг': 'not stated',
  'дефицит бюджета': 'not stated',
  'doc_id': np.int64(1),
  'chunk_id': np.int64(0),
  'original_text': '"Индекс Мосбиржи -3%. Это самое большое дневное падение индекса с 7 октября. #шок_новости На фото - перфоманс акций индекса широкого рынка Мосбиржи за сегодня. "'},
 {'процентная ставка': 'not stated',
  'ВВП': 'not stated',
  'инфляция': 'not stated',
  'безработица': 'not stated',
  'капитал': 'not stated',
  'инвестиции': 'not stated',
  'п

In [63]:
df100 = pd.DataFrame(results)
df100

,процентная ставка,ВВП,инфляция,безработица,капитал,инвестиции,производство,потребление,численность рабочей силы,сбережения,...,импорт,экспорт,государственные расходы,государственный долг,дефицит бюджета,doc_id,chunk_id,original_text,category,details
0,not stated,not stated,not stated,not stated,down,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,1,0,"""Индекс Мосбиржи -3%. Это самое большое дневно...",NaN,NaN
1,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,2,0,"""​​Циан взлетел на апелляции Крупнейший в Росс...",NaN,NaN
2,not stated,down,up,not stated,not stated,not stated,down,not stated,not stated,not stated,...,not stated,down,not stated,not stated,up,2,1,"""С 5 февраля ЕС вводит эмбарго на российские п...",NaN,NaN
3,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,up,3,0,Госбюджет РФ рискует недополучить еще больше н...,NaN,NaN
4,не указано,не указано,не указано,не указано,рост портфеля жилищного кредитования (+3.5%),не указано,не указано,не указано,не указано,не указано,...,не указано,не указано,не указано,не указано,не указано,4,0,"""Новый отчёт Сбера (SBER): чистая прибыль за 1...",NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,not stated,down,not stated,not stated,not stated,down,down,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,51,5,"Коммерческая недвижимость в США и ЕС, которая ...",NaN,NaN
96,not stated,down,not stated,not stated,not stated,not stated,down,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,52,0,Никто из опрошенных теперь не ожидает сильного...,NaN,NaN
97,не указано,не указано,не указано,не указано,не указано,не указано,не указано,не указано,не указано,не указано,...,не указано,не указано,не указано,не указано,не указано,57,1,"""​​ВТБ с рекордными убытками. Бумаги падают, д...",NaN,NaN
98,up,not stated,not stated,not stated,up,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,58,2,При этом регулярная прибыль смогла показать ро...,NaN,NaN


### Видим проблему с промптом, не всегда выдается что надо, но в целом все можно поправить руками

In [ ]:
df100['процентная ставка'].value_counts()

процентная ставка
not stated    70
up            10
down           8
не указано     7
Name: count, dtype: int64

In [ ]:
df100['капитал'].value_counts()

капитал
not stated                                      70
up                                              12
down                                             7
не указано                                       5
рост портфеля жилищного кредитования (+3.5%)     1
Name: count, dtype: int64

## 2-АЯ ЧАСТЬ РАЗМЕТКИ(ОТ 100 ДО 449)

In [ ]:
print("🚀 Начинаю разметку через GigaChat...")
dataset_100_449 = list(data['chunk_text'])[100:449]
results_2 = []

# verify_ssl_certs=False критичен, если у вас нет сертификатов Минцифры
with GigaChat(credentials=AUTH_KEY, verify_ssl_certs=False) as giga:
    for i, text in enumerate(dataset_100_449, 1):
        print(f'Обработка документа {i + 100}...')
        label = label_data(text, giga)
        label['doc_id'] = data.iloc[i + 100, 0]
        label['chunk_id'] = data.iloc[i + 100, 1]
        label['original_text'] = text

        results_2.append(label)

🚀 Начинаю разметку через GigaChat...
Обработка документа 101...
Обработка документа 102...
Обработка документа 103...
Обработка документа 104...
Обработка документа 105...
Обработка документа 106...
Обработка документа 107...
Обработка документа 108...
Обработка документа 109...
Обработка документа 110...
Обработка документа 111...
Обработка документа 112...
Обработка документа 113...
Обработка документа 114...
Обработка документа 115...
Обработка документа 116...
Обработка документа 117...
Обработка документа 118...
Обработка документа 119...
Обработка документа 120...
Обработка документа 121...
Обработка документа 122...
Обработка документа 123...
Обработка документа 124...
Обработка документа 125...
Обработка документа 126...
Обработка документа 127...
Обработка документа 128...
Обработка документа 129...
Обработка документа 130...
Обработка документа 131...
Обработка документа 132...
Обработка документа 133...
Обработка документа 134...
Обработка документа 135...
Обработка документ

KeyboardInterrupt: 

In [ ]:
df449 = pd.DataFrame(results_2)
df449_full = pd.concat([df100, df449], axis=0)

In [78]:
df449_full_new = df449_full.drop(['category', 'details', 'государственные доходы', 'налоговые поступления'], axis=1)

In [312]:
## -30 ДАННЫХ
df449_full_good = df449_full_new.dropna()

## НЕМНОГО ПОФИКСИМ ОТВЕТ

In [313]:
# процентная ставка
df449_full_good['процентная ставка'].value_counts()

процентная ставка
not stated                    311
up                             67
down                           24
не указано                     18
up or down (не определено)      1
Name: count, dtype: int64

In [314]:
df449_full_good.loc[df449_full_good['процентная ставка'] == 'не указано', 'процентная ставка'] = 'not stated'
df449_full_good.loc[df449_full_good['процентная ставка'] == 'up or down (не определено)', 'процентная ставка'] = 'not stated'


In [315]:
df449_full_good['процентная ставка'].value_counts()

процентная ставка
not stated    330
up             67
down           24
Name: count, dtype: int64

In [316]:
## ВВП
df449_full_good['ВВП'].value_counts()

ВВП
not stated    323
down           65
не указано     18
up             15
Name: count, dtype: int64

In [317]:
df449_full_good.loc[df449_full_good['ВВП'] == 'не указано', 'ВВП'] = 'not stated'

In [318]:
df449_full_good['ВВП'].value_counts()

ВВП
not stated    341
down           65
up             15
Name: count, dtype: int64

In [319]:
## безработица
df449_full_good['безработица'].value_counts()

безработица
not stated    391
не указано     19
up              8
down            3
Name: count, dtype: int64

In [320]:
df449_full_good.loc[df449_full_good['безработица'] == 'не указано', 'безработица'] = 'not stated'

In [321]:
df449_full_good['безработица'].value_counts()

безработица
not stated    410
up              8
down            3
Name: count, dtype: int64

In [322]:
## капитал
df449_full_good['капитал'].value_counts()

капитал
not stated                                             343
up                                                      37
down                                                    24
не указано                                              15
рост портфеля жилищного кредитования (+3.5%)             1
up» (благодаря запасу капитала банковского сектора)      1
Name: count, dtype: int64

In [ ]:
df449_full_good.loc[df449_full_good['капитал'] == 'не указано', 'капитал'] = 'not stated'

df449_full_good.loc[df449_full_good['капитал'] == 'рост портфеля жилищного кредитования (+3.5%)', 'капитал'] = 'up'
df449_full_good.loc[df449_full_good['капитал'] == 'up» (благодаря запасу капитала банковского сектора)', 'капитал'] = 'up'


In [324]:
df449_full_good['капитал'].value_counts()

капитал
not stated    358
up             39
down           24
Name: count, dtype: int64

In [325]:
# инвестиции
df449_full_good['инвестиции'].value_counts()

инвестиции
not stated    350
down           27
up             25
не указано     19
Name: count, dtype: int64

In [326]:
df449_full_good.loc[df449_full_good['инвестиции'] == 'не указано', 'инвестиции'] = 'not stated'

In [327]:
df449_full_good['инвестиции'].value_counts()

инвестиции
not stated    369
down           27
up             25
Name: count, dtype: int64

In [328]:
# производство
df449_full_good['производство'].value_counts()

производство
not stated                          300
down                                 72
up                                   30
не указано                           18
down», (электростанции РусГидро)      1
Name: count, dtype: int64

In [329]:
df449_full_good.loc[df449_full_good['производство'] == 'не указано', 'производство'] = 'not stated'
df449_full_good.loc[df449_full_good['производство'] == 'down», (электростанции РусГидро)', 'производство'] = 'down'


In [ ]:
df449_full_good['производство'].value_counts()

производство
not stated    318
down           73
up             30
Name: count, dtype: int64

In [331]:
# потребление
df449_full_good['потребление'].value_counts()

потребление
not stated    382
не указано     18
down           13
up              8
Name: count, dtype: int64

In [332]:
df449_full_good.loc[df449_full_good['потребление'] == 'не указано', 'потребление'] = 'not stated'

In [333]:
df449_full_good['потребление'].value_counts()

потребление
not stated    400
down           13
up              8
Name: count, dtype: int64

In [334]:
# численность рабочей силы
df449_full_good['численность рабочей силы'].value_counts()

численность рабочей силы
not stated    393
не указано     19
down            5
up              4
Name: count, dtype: int64

In [335]:
df449_full_good.loc[df449_full_good['численность рабочей силы'] == 'не указано', 'численность рабочей силы'] = 'not stated'

In [336]:
df449_full_good['численность рабочей силы'].value_counts()

численность рабочей силы
not stated    412
down            5
up              4
Name: count, dtype: int64

In [337]:
# сбережения
df449_full_good['сбережения'].value_counts()

сбережения
not stated    396
не указано     19
down            3
up              3
Name: count, dtype: int64

In [338]:
df449_full_good.loc[df449_full_good['сбережения'] == 'не указано', 'сбережения'] = 'not stated'

In [339]:
df449_full_good['сбережения'].value_counts()

сбережения
not stated    415
down            3
up              3
Name: count, dtype: int64

In [340]:
# заработные платы
df449_full_good['заработные платы'].value_counts()

заработные платы
not stated    383
не указано     19
down           11
up              8
Name: count, dtype: int64

In [341]:
df449_full_good.loc[df449_full_good['заработные платы'] == 'не указано', 'заработные платы'] = 'not stated'

In [342]:
df449_full_good['заработные платы'].value_counts()

заработные платы
not stated    402
down           11
up              8
Name: count, dtype: int64

In [343]:
# доходы населения
df449_full_good['доходы населения'].value_counts()

доходы населения
not stated                                      374
down                                             18
не указано                                       18
up                                               10
не явно увеличатся, но не уточнено конкретно      1
Name: count, dtype: int64

In [344]:
df449_full_good.loc[df449_full_good['доходы населения'] == 'не указано', 'доходы населения'] = 'not stated'
df449_full_good.loc[df449_full_good['доходы населения'].apply(lambda x: x.split()[0]) == 'не', 'доходы населения'] = 'up'

In [345]:
df449_full_good['доходы населения'].value_counts()

доходы населения
not stated    392
down           18
up             11
Name: count, dtype: int64

In [346]:
# валютный курс
df449_full_good['валютный курс'].value_counts()

валютный курс
not stated                   313
down                          49
up                            42
не указано                    16
up», (доллар укрепляется)      1
Name: count, dtype: int64

In [347]:
df449_full_good.loc[df449_full_good['валютный курс'] == 'не указано', 'валютный курс'] = 'not stated'
df449_full_good.loc[df449_full_good['валютный курс'] == 'up», (доллар укрепляется)', 'валютный курс'] = 'up'


In [348]:
df449_full_good['валютный курс'].value_counts()

валютный курс
not stated    329
down           49
up             43
Name: count, dtype: int64

In [349]:
# импорт
df449_full_good['импорт'].value_counts()

импорт
not stated    366
down           24
не указано     19
up             12
Name: count, dtype: int64

In [350]:
df449_full_good.loc[df449_full_good['импорт'] == 'не указано', 'импорт'] = 'not stated'

In [351]:
df449_full_good['импорт'].value_counts()

импорт
not stated    385
down           24
up             12
Name: count, dtype: int64

In [352]:
# экспорт
df449_full_good['экспорт'].value_counts()

экспорт
not stated                               308
up                                        55
down                                      38
не указано                                18
up»,                                       1
up» (азотные удобрения, экспорт в ЕС)      1
Name: count, dtype: int64

In [353]:
df449_full_good.loc[df449_full_good['экспорт'] == 'не указано', 'экспорт'] = 'not stated'
df449_full_good.loc[df449_full_good['экспорт'] == 'up» (азотные удобрения, экспорт в ЕС)', 'экспорт'] = 'up'
df449_full_good.loc[df449_full_good['экспорт'] == 'up»,', 'экспорт'] = 'up'


In [354]:
df449_full_good['экспорт'].value_counts()

экспорт
not stated    326
up             57
down           38
Name: count, dtype: int64

In [355]:
# государственные расходы
df449_full_good['государственные расходы'].value_counts()

государственные расходы
not stated    395
не указано     19
up              7
Name: count, dtype: int64

In [356]:
df449_full_good.loc[df449_full_good['государственные расходы'] == 'не указано', 'государственные расходы'] = 'not stated'

In [357]:
df449_full_good['государственные расходы'].value_counts()

государственные расходы
not stated    414
up              7
Name: count, dtype: int64

In [ ]:
# государственный долг
df449_full_good['государственный долг'].value_counts()

государственный долг
not stated    387
не указано     19
up             14
down            1
Name: count, dtype: int64

In [359]:
df449_full_good.loc[df449_full_good['государственный долг'] == 'не указано', 'государственный долг'] = 'not stated'

In [360]:
df449_full_good['государственный долг'].value_counts()

государственный долг
not stated    406
up             14
down            1
Name: count, dtype: int64

In [361]:
# дефицит бюджета
df449_full_good['дефицит бюджета'].value_counts()

дефицит бюджета
not stated    378
up             19
не указано     19
down            5
Name: count, dtype: int64

In [362]:
df449_full_good.loc[df449_full_good['дефицит бюджета'] == 'не указано', 'дефицит бюджета'] = 'not stated'

In [363]:
df449_full_good['дефицит бюджета'].value_counts()

дефицит бюджета
not stated    397
up             19
down            5
Name: count, dtype: int64

In [364]:
## инфляция
df449_full_good['инфляция'].value_counts()

инфляция
not stated    345
up             47
не указано     18
down           11
Name: count, dtype: int64

In [365]:
df449_full_good.loc[df449_full_good['инфляция'] == 'не указано', 'инфляция'] = 'not stated'

In [366]:
df449_full_good['инфляция'].value_counts()

инфляция
not stated    363
up             47
down           11
Name: count, dtype: int64

In [367]:
df449_full_good.to_csv('df449_full_good.csv', index=False)

In [ ]:
#df449_full.to_csv('siksnmdc.csv', index=False)

## 3-АЯ ЧАСТЬ РАЗМЕТКИ(НА АККАУНТЕ КОНЧИЛИСЬ ТОКЕНЫ, ВЗЯЛИ НОВЫЙ) - c 449 до 1000

In [ ]:
AUTH_KEY = "=="

In [371]:
print("🚀 Начинаю разметку через GigaChat...")
dataset_449_1000 = list(data['chunk_text'])[449:1000]
results_3 = []

# verify_ssl_certs=False критичен, если у вас нет сертификатов Минцифры
with GigaChat(credentials=AUTH_KEY, verify_ssl_certs=False) as giga:
    for i, text in enumerate(dataset_449_1000, 1):
        print(f'Обработка документа {i + 449}...')
        label = label_data(text, giga)
        label['doc_id'] = data.iloc[i + 449, 0]
        label['chunk_id'] = data.iloc[i + 449, 1]
        label['original_text'] = text

        results_3.append(label)

🚀 Начинаю разметку через GigaChat...
Обработка документа 450...
Обработка документа 451...
Обработка документа 452...
Обработка документа 453...
Обработка документа 454...
Обработка документа 455...
Обработка документа 456...
Обработка документа 457...
Обработка документа 458...
Обработка документа 459...
Обработка документа 460...
Обработка документа 461...
Обработка документа 462...
Обработка документа 463...
Обработка документа 464...
Обработка документа 465...
Обработка документа 466...
Обработка документа 467...
Обработка документа 468...
Обработка документа 469...
Обработка документа 470...
Обработка документа 471...
Обработка документа 472...
Обработка документа 473...
Обработка документа 474...
Обработка документа 475...
Обработка документа 476...
Обработка документа 477...
Обработка документа 478...
Обработка документа 479...
Обработка документа 480...
Обработка документа 481...
Обработка документа 482...
Обработка документа 483...
Обработка документа 484...
Обработка документ

In [448]:
df1000

,процентная ставка,ВВП,инфляция,безработица,капитал,инвестиции,производство,потребление,численность рабочей силы,сбережения,...,доходы населения,валютный курс,импорт,экспорт,государственные расходы,государственный долг,дефицит бюджета,doc_id,chunk_id,original_text
0,не указано,не указано,не указано,не указано,не указано,не указано,не указано,не указано,не указано,не указано,...,не указано,не указано,не указано,не указано,не указано,не указано,не указано,326,3,Благодаря текущему оптимизму на американском р...
1,up,not stated,not stated,not stated,up,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,not stated,327,0,С начала года капитализация Microsoft выросла ...
2,не указано,не указано,не указано,не указано,не указано,не указано,не указано,не указано,не указано,не указано,...,не указано,не указано,не указано,не указано,не указано,не указано,не указано,328,0,"""️️️ Шокирующие прогнозы на 2024 год от Фёдора..."
3,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,not stated,330,0,""" Как закрылась основная торговая сессия на ро..."
4,up,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,not stated,330,2,""" Регуляторы разных стран пытаются остановить ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
546,not stated,down,not stated,not stated,down,down,down,not stated,not stated,not stated,...,not stated,not stated,down,not stated,not stated,not stated,not stated,759,0,"""Великобритания ввела новые санкции. Как реаги..."
547,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,up,not stated,up,760,0,"""Закрываем идею по Транснефти Мы закрываем на..."
548,up,not stated,not stated,down,not stated,not stated,not stated,not stated,up,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,not stated,760,1,"""Рынки снижаются, но Bank of America прогнозир..."
549,not stated,not stated,not stated,down,not stated,not stated,not stated,not stated,up,not stated,...,up,not stated,not stated,not stated,not stated,not stated,not stated,760,2,"По данным Refinitiv, экономисты прогнозируют, ..."


In [449]:
df1000 = pd.DataFrame(results_3)
df1000 = df1000.drop(['category', 'details', 'saving'], axis=1)
df_1000_good = df1000.dropna()

In [450]:
# процентная ставка
df_1000_good['процентная ставка'].value_counts()

процентная ставка
not stated    353
up             97
down           42
не указано     23
Name: count, dtype: int64

In [451]:
df_1000_good.loc[df_1000_good['процентная ставка'] == 'не указано', 'процентная ставка'] = 'not stated'

In [452]:
df_1000_good['процентная ставка'].value_counts()

процентная ставка
not stated    376
up             97
down           42
Name: count, dtype: int64

In [453]:
## ВВП
df_1000_good['ВВП'].value_counts()

ВВП
not stated    424
down           41
не указано     27
up             23
Name: count, dtype: int64

In [454]:
df_1000_good.loc[df_1000_good['ВВП'] == 'не указано', 'ВВП'] = 'not stated'

In [455]:
df_1000_good['ВВП'].value_counts()

ВВП
not stated    451
down           41
up             23
Name: count, dtype: int64

In [456]:
## безработица
df_1000_good['безработица'].value_counts()

безработица
not stated    463
не указано     28
up             13
down           11
Name: count, dtype: int64

In [457]:
df_1000_good.loc[df_1000_good['безработица'] == 'не указано', 'безработица'] = 'not stated'

In [458]:
df_1000_good['безработица'].value_counts()

безработица
not stated    491
up             13
down           11
Name: count, dtype: int64

In [459]:
## капитал
df_1000_good['капитал'].value_counts()

капитал
not stated                        407
up                                 53
down                               29
не указано                         24
up», (рост капитальных затрат)      1
not expanded                        1
Name: count, dtype: int64

In [460]:
df_1000_good.loc[df_1000_good['капитал'] == 'не указано', 'капитал'] = 'not stated'

df_1000_good.loc[df_1000_good['капитал'].apply(lambda x: x.split()[-1]) == 'expanded', 'капитал'] = 'not stated'
df_1000_good.loc[df_1000_good['капитал'] == 'up», (рост капитальных затрат)', 'капитал'] = 'up'

In [461]:
df_1000_good['капитал'].value_counts()

капитал
not stated    432
up             54
down           29
Name: count, dtype: int64

In [462]:
# инвестиции
df_1000_good['инвестиции'].value_counts()

инвестиции
not stated    418
down           36
up             34
не указано     27
Name: count, dtype: int64

In [463]:
df_1000_good.loc[df_1000_good['инвестиции'] == 'не указано', 'инвестиции'] = 'not stated'

In [464]:
df_1000_good['инвестиции'].value_counts()

инвестиции
not stated    445
down           36
up             34
Name: count, dtype: int64

In [ ]:
# производство
df_1000_good['производство'].value_counts()

производство
not stated    391
down           60
up             38
не указано     26
Name: count, dtype: int64

In [466]:
df_1000_good.loc[df_1000_good['производство'] == 'не указано', 'производство'] = 'not stated'

In [467]:
df_1000_good['производство'].value_counts()

производство
not stated    417
down           60
up             38
Name: count, dtype: int64

In [468]:
# потребление
df_1000_good['потребление'].value_counts()

потребление
not stated    451
не указано     28
down           20
up             16
Name: count, dtype: int64

In [469]:
df_1000_good.loc[df_1000_good['потребление'] == 'не указано', 'потребление'] = 'not stated'

In [470]:
df_1000_good['потребление'].value_counts()

потребление
not stated    479
down           20
up             16
Name: count, dtype: int64

In [471]:
# численность рабочей силы
df_1000_good['численность рабочей силы'].value_counts()

численность рабочей силы
not stated    469
не указано     28
down           11
up              7
Name: count, dtype: int64

In [ ]:
df_1000_good.loc[df_1000_good['численность рабочей силы'] == 'не указано', 'численность рабочей силы'] = 'not stated'

In [473]:
df_1000_good['численность рабочей силы'].value_counts()

численность рабочей силы
not stated    497
down           11
up              7
Name: count, dtype: int64

In [474]:
# сбережения
df_1000_good['сбережения'].value_counts()

сбережения
not stated    479
не указано     28
up              6
down            2
Name: count, dtype: int64

In [475]:
df_1000_good.loc[df_1000_good['сбережения'] == 'не указано', 'сбережения'] = 'not stated'

In [476]:
df_1000_good['сбережения'].value_counts()

сбережения
not stated    507
up              6
down            2
Name: count, dtype: int64

In [477]:
# заработные платы
df_1000_good['заработные платы'].value_counts()

заработные платы
not stated    456
не указано     28
down           18
up             13
Name: count, dtype: int64

In [512]:
df_1000_good.loc[df_1000_good['заработные платы'] == 'не указано', 'заработные платы'] = 'not stated'

In [479]:
df_1000_good['заработные платы'].value_counts()

заработные платы
not stated    484
down           18
up             13
Name: count, dtype: int64

In [480]:
# доходы населения
df_1000_good['доходы населения'].value_counts()

доходы населения
not stated    439
не указано     28
up             24
down           24
Name: count, dtype: int64

In [481]:
df_1000_good.loc[df_1000_good['доходы населения'] == 'не указано', 'доходы населения'] = 'not stated'
df_1000_good.loc[df_1000_good['доходы населения'].apply(lambda x: x.split()[0]) == 'не', 'доходы населения'] = 'up'

In [482]:
df_1000_good['доходы населения'].value_counts()


доходы населения
not stated    467
up             24
down           24
Name: count, dtype: int64

In [483]:
# валютный курс
df_1000_good['валютный курс'].value_counts()

валютный курс
not stated                               397
up                                        54
down                                      35
не указано                                23
up-down                                    3
up/down                                    1
не указано (зависит от динамики)           1
up (краткосрочно), down (долгосрочно)      1
Name: count, dtype: int64

In [485]:
df_1000_good.loc[df_1000_good['валютный курс'] == 'не указано', 'валютный курс'] = 'not stated'
df_1000_good.loc[df_1000_good['валютный курс'] == 'не указано (зависит от динамики)', 'валютный курс'] = 'not stated'

df_1000_good.loc[df_1000_good['валютный курс'] == 'up/down', 'валютный курс'] = 'up'
df_1000_good.loc[df_1000_good['валютный курс'] == 'up-down', 'валютный курс'] = 'up'
df_1000_good.loc[df_1000_good['валютный курс'] == 'up (краткосрочно), down (долгосрочно)', 'валютный курс'] = 'up'


In [486]:
df_1000_good['валютный курс'].value_counts()

валютный курс
not stated    421
up             59
down           35
Name: count, dtype: int64

In [487]:
# импорт
df_1000_good['импорт'].value_counts()

импорт
not stated    453
не указано     28
down           24
up             10
Name: count, dtype: int64

In [488]:
df_1000_good.loc[df_1000_good['импорт'] == 'не указано', 'импорт'] = 'not stated'

In [489]:
df_1000_good['импорт'].value_counts()

импорт
not stated    481
down           24
up             10
Name: count, dtype: int64

In [490]:
# экспорт
df_1000_good['экспорт'].value_counts()

экспорт
not stated    401
up             56
down           31
не указано     27
Name: count, dtype: int64

In [491]:
df_1000_good.loc[df_1000_good['экспорт'] == 'не указано', 'экспорт'] = 'not stated'

In [ ]:
df_1000_good['экспорт'].value_counts()

экспорт
not stated    428
up             56
down           31
Name: count, dtype: int64

In [493]:
# государственные расходы
df_1000_good['государственные расходы'].value_counts()

государственные расходы
not stated    475
не указано     27
up             11
down            1
neutral         1
Name: count, dtype: int64

In [494]:
df_1000_good.loc[df_1000_good['государственные расходы'] == 'не указано', 'государственные расходы'] = 'not stated'

df_1000_good.loc[df_1000_good['государственные расходы'] == 'neutral', 'государственные расходы'] = 'not stated'

In [495]:
df_1000_good['государственные расходы'].value_counts()

государственные расходы
not stated    503
up             11
down            1
Name: count, dtype: int64

In [496]:
# государственный долг
df_1000_good['государственный долг'].value_counts()

государственный долг
not stated    471
не указано     27
up             15
down            1
неизменен       1
Name: count, dtype: int64

In [497]:
df_1000_good.loc[df_1000_good['государственный долг'] == 'не указано', 'государственный долг'] = 'not stated'
df_1000_good.loc[df_1000_good['государственный долг'] == 'неизменен', 'государственный долг'] = 'not stated'

In [498]:
df_1000_good['государственный долг'].value_counts()

государственный долг
not stated    499
up             15
down            1
Name: count, dtype: int64

In [499]:
# дефицит бюджета
df_1000_good['дефицит бюджета'].value_counts()

дефицит бюджета
not stated    467
не указано     27
up             19
down            1
neutral         1
Name: count, dtype: int64

In [500]:
df_1000_good.loc[df_1000_good['дефицит бюджета'] == 'не указано', 'дефицит бюджета'] = 'not stated'
df_1000_good.loc[df_1000_good['дефицит бюджета'] == 'neutral', 'дефицит бюджета'] = 'not stated'

In [501]:
df_1000_good['дефицит бюджета'].value_counts()

дефицит бюджета
not stated    495
up             19
down            1
Name: count, dtype: int64

In [502]:
## инфляция
df_1000_good['инфляция'].value_counts()

инфляция
not stated                  415
up                           50
не указано                   27
down                         22
up (по словам чиновника)      1
Name: count, dtype: int64

In [503]:
df_1000_good.loc[df_1000_good['инфляция'] == 'не указано', 'инфляция'] = 'not stated'
df_1000_good.loc[df_1000_good['инфляция'] == 'up (по словам чиновника)', 'инфляция'] = 'up'


In [504]:
df_1000_good['инфляция'].value_counts()

инфляция
not stated    442
up             51
down           22
Name: count, dtype: int64

In [508]:
df_1000_all = pd.concat([df449_full_good, df_1000_good], axis=0)
df_1000_all

,процентная ставка,ВВП,инфляция,безработица,капитал,инвестиции,производство,потребление,численность рабочей силы,сбережения,...,доходы населения,валютный курс,импорт,экспорт,государственные расходы,государственный долг,дефицит бюджета,doc_id,chunk_id,original_text
0,not stated,not stated,not stated,not stated,down,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,not stated,1,0,"""Индекс Мосбиржи -3%. Это самое большое дневно..."
1,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,not stated,2,0,"""​​Циан взлетел на апелляции Крупнейший в Росс..."
2,not stated,down,up,not stated,not stated,not stated,down,not stated,not stated,not stated,...,down,not stated,not stated,down,not stated,not stated,up,2,1,"""С 5 февраля ЕС вводит эмбарго на российские п..."
3,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,up,3,0,Госбюджет РФ рискует недополучить еще больше н...
4,not stated,not stated,not stated,not stated,up,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,not stated,4,0,"""Новый отчёт Сбера (SBER): чистая прибыль за 1..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
546,not stated,down,not stated,not stated,down,down,down,not stated,not stated,not stated,...,not stated,not stated,down,not stated,not stated,not stated,not stated,759,0,"""Великобритания ввела новые санкции. Как реаги..."
547,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,up,not stated,up,760,0,"""Закрываем идею по Транснефти Мы закрываем на..."
548,up,not stated,not stated,down,not stated,not stated,not stated,not stated,up,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,not stated,760,1,"""Рынки снижаются, но Bank of America прогнозир..."
549,not stated,not stated,not stated,down,not stated,not stated,not stated,not stated,up,not stated,...,up,not stated,not stated,not stated,not stated,not stated,not stated,760,2,"По данным Refinitiv, экономисты прогнозируют, ..."


In [509]:
df_1000_all.to_csv('df_1000_all_good.csv', index=False)

## СПОРТСМЕНЫ НА МЕСТЕ? РАЗМЕЧАЕМ С 1000 ДО 2000

In [511]:
print("🚀 Начинаю разметку через GigaChat...")
dataset_1000_end = list(data['chunk_text'])[1000:]
results_4 = []

# verify_ssl_certs=False критичен, если у вас нет сертификатов Минцифры
with GigaChat(credentials=AUTH_KEY, verify_ssl_certs=False) as giga:
    for i, text in enumerate(dataset_1000_end, 1):
        print(f'Обработка документа {i + 1000}...')
        label = label_data(text, giga)
        label['doc_id'] = data.iloc[i + 1000, 0]
        label['chunk_id'] = data.iloc[i + 1000, 1]
        label['original_text'] = text

        results_4.append(label)

🚀 Начинаю разметку через GigaChat...
Обработка документа 1001...
Обработка документа 1002...
Обработка документа 1003...
Обработка документа 1004...
Обработка документа 1005...
Обработка документа 1006...
Обработка документа 1007...
Обработка документа 1008...
Обработка документа 1009...
Обработка документа 1010...
Обработка документа 1011...
Обработка документа 1012...
Обработка документа 1013...
Обработка документа 1014...
Обработка документа 1015...
Обработка документа 1016...
Обработка документа 1017...
Обработка документа 1018...
Обработка документа 1019...
Обработка документа 1020...
Обработка документа 1021...
Обработка документа 1022...
Обработка документа 1023...
Обработка документа 1024...
Обработка документа 1025...
Обработка документа 1026...
Обработка документа 1027...
Обработка документа 1028...
Обработка документа 1029...
Обработка документа 1030...
Обработка документа 1031...
Обработка документа 1032...
Обработка документа 1033...
Обработка документа 1034...
Обработка д

IndexError: index 1954 is out of bounds for axis 0 with size 1954

In [514]:
df2000

,процентная ставка,ВВП,инфляция,безработица,капитал,инвестиции,производство,потребление,численность рабочей силы,сбережения,...,импорт,экспорт,государственные расходы,государственный долг,дефицит бюджета,doc_id,chunk_id,original_text,category,details
0,not stated,not stated,up,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,762,0,"""Среднегодовая доходность активов в России за ...",NaN,NaN
1,up,not stated,down,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,762,1,"""Пауэлл не смог напугать инвесторов, а нефть д...",NaN,NaN
2,up,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,762,2,"Между тем, участники рынка с 20%-й вероятность...",NaN,NaN
3,not stated,not stated,not stated,not stated,down,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,762,3,Эти стимулирующие меры направлены на оживление...,NaN,NaN
4,не указано,не указано,не указано,не указано,не указано,не указано,не указано,не указано,не указано,не указано,...,не указано,не указано,не указано,не указано,не указано,762,4,За тот же период выручка компании составила 12...,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
948,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,1491,0,В начале октября валютный курс тестирует сопро...,NaN,NaN
949,not stated,down,not stated,not stated,not stated,not stated,down,not stated,not stated,not stated,...,up,down,not stated,not stated,not stated,1491,4,"""Финансовый конгресс Банка России: что важного...",NaN,NaN
950,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1492,0,Этот хаб можно расположить в одной из стран Пе...,Ошибка,Expecting value: line 1 column 1 (char 0)
951,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1495,2,"""За последние 2 недели рубль ослаб почти на 8%...",NaN,NaN


In [602]:
df2000 = pd.DataFrame(results_4)
df2000 = df2000.drop(['category', 'details'], axis=1)
df_2000_good = df2000.dropna()


In [603]:
df_2000_good.to_csv('df_2000_good.csv', index=False)

In [604]:
# процентная ставка
df_2000_good['процентная ставка'].value_counts()

процентная ставка
not stated                    573
up                            192
не указано                     53
down                           51
up or down (неопределенно)      1
Name: count, dtype: int64

In [605]:
df_2000_good.loc[df_2000_good['процентная ставка'] == 'не указано', 'процентная ставка'] = 'not stated'
df_2000_good.loc[df_2000_good['процентная ставка'] == 'up or down (неопределенно)', 'процентная ставка'] = 'not stated'

In [606]:
df_2000_good['процентная ставка'].value_counts()

процентная ставка
not stated    627
up            192
down           51
Name: count, dtype: int64

In [607]:
## ВВП
df_2000_good['ВВП'].value_counts()

ВВП
not stated    678
down           99
не указано     50
up             43
Name: count, dtype: int64

In [608]:
df_2000_good.loc[df_2000_good['ВВП'] == 'не указано', 'ВВП'] = 'not stated'

In [609]:
df_2000_good['ВВП'].value_counts()

ВВП
not stated    728
down           99
up             43
Name: count, dtype: int64

In [611]:
## безработица
df_2000_good['безработица'].value_counts()

безработица
not stated    801
не указано     54
up             10
down            5
Name: count, dtype: int64

In [612]:
df_2000_good.loc[df_2000_good['безработица'] == 'не указано', 'безработица'] = 'not stated'

In [613]:
df_2000_good['безработица'].value_counts()

безработица
not stated    855
up             10
down            5
Name: count, dtype: int64

In [614]:
## капитал
df_2000_good['капитал'].value_counts()

капитал
not stated                                                                          680
up                                                                                   75
down                                                                                 65
не указано                                                                           48
up», поскольку сделка увеличит долю РусАгро на рынке и расширит географию продаж      1
down»,                                                                                1
Name: count, dtype: int64

In [615]:
df_2000_good.loc[df_2000_good['капитал'] == 'не указано', 'капитал'] = 'not stated'

df_2000_good.loc[df_2000_good['капитал'] == 'down»,', 'капитал'] = 'down'
df_2000_good.loc[df_2000_good['капитал'] == 'up», поскольку сделка увеличит долю РусАгро на рынке и расширит географию продаж', 'капитал'] = 'up'

In [616]:
df_2000_good['капитал'].value_counts()

капитал
not stated    728
up             76
down           66
Name: count, dtype: int64

In [617]:
df_2000_good['капитал'].value_counts()

капитал
not stated    728
up             76
down           66
Name: count, dtype: int64

In [618]:
# инвестиции
df_2000_good['инвестиции'].value_counts()

инвестиции
not stated                                                                                                 711
down                                                                                                        68
не указано                                                                                                  51
up                                                                                                          38
down», возможное up                                                                                          1
up», так как приобретение требует капитальных вложений и расширения производственных мощностей компании      1
Name: count, dtype: int64

In [619]:
df_2000_good.loc[df_2000_good['инвестиции'] == 'не указано', 'инвестиции'] = 'not stated'

df_2000_good.loc[df_2000_good['инвестиции'] == 'down», возможное up', 'инвестиции'] = 'down'
df_2000_good.loc[df_2000_good['инвестиции'] == 'up», так как приобретение требует капитальных вложений и расширения производственных мощностей компании', 'инвестиции'] = 'up'

In [620]:
df_2000_good['инвестиции'].value_counts()

инвестиции
not stated    762
down           69
up             39
Name: count, dtype: int64

In [ ]:
# производство
df_2000_good['производство'].value_counts()

производство
not stated                        646
down                              111
up                                 58
не указано                         54
up», (рост цен на нефть и газ)      1
Name: count, dtype: int64

In [622]:
df_2000_good.loc[df_2000_good['производство'] == 'не указано', 'производство'] = 'not stated'
df_2000_good.loc[df_2000_good['производство'] == 'up», (рост цен на нефть и газ)', 'производство'] = 'up'

In [623]:
df_2000_good['производство'].value_counts()

производство
not stated    700
down          111
up             59
Name: count, dtype: int64

In [624]:
# потребление
df_2000_good['потребление'].value_counts()

потребление
not stated    753
не указано     54
down           43
up             20
Name: count, dtype: int64

In [625]:
df_2000_good.loc[df_2000_good['потребление'] == 'не указано', 'потребление'] = 'not stated'

In [626]:
df_2000_good['потребление'].value_counts()

потребление
not stated    807
down           43
up             20
Name: count, dtype: int64

In [627]:
# численность рабочей силы
df_2000_good['численность рабочей силы'].value_counts()

численность рабочей силы
not stated    804
не указано     55
down            8
up              3
Name: count, dtype: int64

In [628]:
df_2000_good.loc[df_2000_good['численность рабочей силы'] == 'не указано', 'численность рабочей силы'] = 'not stated'

In [629]:
df_2000_good['численность рабочей силы'].value_counts()


численность рабочей силы
not stated    859
down            8
up              3
Name: count, dtype: int64

In [630]:
# сбережения
df_2000_good['сбережения'].value_counts()

сбережения
not stated    790
не указано     55
up             19
down            6
Name: count, dtype: int64

In [631]:
df_2000_good.loc[df_2000_good['сбережения'] == 'не указано', 'сбережения'] = 'not stated'

In [632]:
df_2000_good['сбережения'].value_counts()

сбережения
not stated    845
up             19
down            6
Name: count, dtype: int64

In [633]:
# заработные платы
df_2000_good['заработные платы'].value_counts()

заработные платы
not stated    782
не указано     55
down           23
up             10
Name: count, dtype: int64

In [634]:
df_2000_good.loc[df_2000_good['заработные платы'] == 'не указано', 'заработные платы'] = 'not stated'

In [635]:
df_2000_good['заработные платы'].value_counts()


заработные платы
not stated    837
down           23
up             10
Name: count, dtype: int64

In [637]:
# доходы населения
df_2000_good['доходы населения'].value_counts()

доходы населения
not stated    752
не указано     55
down           45
up             18
Name: count, dtype: int64

In [638]:
df_2000_good.loc[df_2000_good['доходы населения'] == 'не указано', 'доходы населения'] = 'not stated'

In [639]:
df_2000_good['доходы населения'].value_counts()

доходы населения
not stated    807
down           45
up             18
Name: count, dtype: int64

In [640]:
# валютный курс
df_2000_good['валютный курс'].value_counts()

валютный курс
not stated                                               651
up                                                       104
down                                                      61
не указано                                                48
parity reached                                             1
не определено»                                             1
up» (рубль ослабевает)                                     1
up» (курс рубля укрепляется относительно евро и юаня)      1
mixed                                                      1
не указано, вероятно down                                  1
Name: count, dtype: int64

In [641]:
df_2000_good.loc[df_2000_good['валютный курс'] == 'не указано', 'валютный курс'] = 'not stated'
df_2000_good.loc[df_2000_good['валютный курс'] == 'не определено»', 'валютный курс'] = 'not stated'
df_2000_good.loc[df_2000_good['валютный курс'] == 'не указано, вероятно down', 'валютный курс'] = 'not stated'
df_2000_good.loc[df_2000_good['валютный курс'] == 'parity reached', 'валютный курс'] = 'not stated'
df_2000_good.loc[df_2000_good['валютный курс'] == 'mixed', 'валютный курс'] = 'not stated'

df_2000_good.loc[df_2000_good['валютный курс'] == 'up» (рубль ослабевает)', 'валютный курс'] = 'up'
df_2000_good.loc[df_2000_good['валютный курс'] == 'up» (курс рубля укрепляется относительно евро и юаня)', 'валютный курс'] = 'down'

In [642]:
df_2000_good['валютный курс'].value_counts()

валютный курс
not stated    703
up            105
down           62
Name: count, dtype: int64

In [643]:
# импорт
df_2000_good['импорт'].value_counts()

импорт
not stated    733
down           55
не указано     54
up             28
Name: count, dtype: int64

In [644]:
df_2000_good.loc[df_2000_good['импорт'] == 'не указано', 'импорт'] = 'not stated'

In [645]:
df_2000_good['импорт'].value_counts()

импорт
not stated    787
down           55
up             28
Name: count, dtype: int64

In [646]:
# экспорт
df_2000_good['экспорт'].value_counts()

экспорт
not stated                                                                                          674
up                                                                                                   76
down                                                                                                 62
не указано                                                                                           54
не указано, возможно down                                                                             1
up» (рост экспортных цен на сырье)                                                                    1
up», расширение доли на международном рынке благодаря усилению присутствия на масложировом рынке      1
up (кратковременно)                                                                                   1
Name: count, dtype: int64

In [647]:
df_2000_good.loc[df_2000_good['экспорт'] == 'не указано', 'экспорт'] = 'not stated'
df_2000_good.loc[df_2000_good['экспорт'] == 'не указано, возможно down', 'экспорт'] = 'not stated'


df_2000_good.loc[df_2000_good['экспорт'] == 'up» (рост экспортных цен на сырье)'] = 'up'
df_2000_good.loc[df_2000_good['экспорт'] == 'up», расширение доли на международном рынке благодаря усилению присутствия на масложировом рынке'] = 'up'
df_2000_good.loc[df_2000_good['экспорт'] == 'up (кратковременно)'] = 'up'


In [ ]:
df_2000_good['экспорт'].value_counts()

экспорт
not stated    729
up             79
down           62
Name: count, dtype: int64

In [649]:
# государственные расходы
df_2000_good['государственные расходы'].value_counts()

государственные расходы
not stated            789
не указано             54
up                     25
некоторое снижение      1
down                    1
Name: count, dtype: int64

In [650]:
df_2000_good.loc[df_2000_good['государственные расходы'] == 'не указано', 'государственные расходы'] = 'not stated'
df_2000_good.loc[df_2000_good['государственные расходы'] == 'некоторое снижение', 'государственные расходы'] = 'down'

In [651]:
df_2000_good['государственные расходы'].value_counts()

государственные расходы
not stated    843
up             25
down            2
Name: count, dtype: int64

In [652]:
# государственный долг
df_2000_good['государственный долг'].value_counts()

государственный долг
not stated    787
не указано     54
up             25
down            4
Name: count, dtype: int64

In [653]:
df_2000_good.loc[df_2000_good['государственный долг'] == 'не указано', 'государственный долг'] = 'not stated'

In [654]:
df_2000_good['государственный долг'].value_counts()

государственный долг
not stated    841
up             25
down            4
Name: count, dtype: int64

In [655]:
# дефицит бюджета
df_2000_good['дефицит бюджета'].value_counts()

дефицит бюджета
not stated    774
не указано     54
up             38
down            3
сокращение      1
Name: count, dtype: int64

In [656]:
df_2000_good.loc[df_2000_good['дефицит бюджета'] == 'не указано', 'дефицит бюджета'] = 'not stated'
df_2000_good.loc[df_2000_good['дефицит бюджета'] == 'сокращение', 'дефицит бюджета'] = 'down'


In [657]:
df_2000_good['дефицит бюджета'].value_counts()

дефицит бюджета
not stated    828
up             38
down            4
Name: count, dtype: int64

In [658]:
## инфляция
df_2000_good['инфляция'].value_counts()

инфляция
not stated    654
up            117
не указано     50
down           49
Name: count, dtype: int64

In [659]:
df_2000_good.loc[df_2000_good['инфляция'] == 'не указано', 'инфляция'] = 'not stated'

In [660]:
df_2000_good['инфляция'].value_counts()

инфляция
not stated    704
up            117
down           49
Name: count, dtype: int64

In [661]:
df_2000_good.shape

(870, 21)

In [662]:
df_1000_good.shape

(515, 21)

In [663]:
df_2000_all = pd.concat([df_1000_all, df_2000_good], axis=0)
df_2000_all

,процентная ставка,ВВП,инфляция,безработица,капитал,инвестиции,производство,потребление,численность рабочей силы,сбережения,...,доходы населения,валютный курс,импорт,экспорт,государственные расходы,государственный долг,дефицит бюджета,doc_id,chunk_id,original_text
0,not stated,not stated,not stated,not stated,down,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,not stated,1,0,"""Индекс Мосбиржи -3%. Это самое большое дневно..."
1,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,not stated,2,0,"""​​Циан взлетел на апелляции Крупнейший в Росс..."
2,not stated,down,up,not stated,not stated,not stated,down,not stated,not stated,not stated,...,down,not stated,not stated,down,not stated,not stated,up,2,1,"""С 5 февраля ЕС вводит эмбарго на российские п..."
3,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,up,3,0,Госбюджет РФ рискует недополучить еще больше н...
4,not stated,not stated,not stated,not stated,up,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,not stated,4,0,"""Новый отчёт Сбера (SBER): чистая прибыль за 1..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
945,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,not stated,1488,0,• Объем торгов металлами в феврале составил 61...
946,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,not stated,1489,0,"""​​Страхованию инвестиций на ИИС — быть В дека..."
947,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,up,not stated,not stated,not stated,not stated,not stated,1489,1,"""​​Что означает 100 рублей за доллар для префо..."
948,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,up,not stated,not stated,not stated,not stated,not stated,1491,0,В начале октября валютный курс тестирует сопро...


In [664]:
df_2000_all.to_csv('df_2000_all_new.csv', index=False)